# Purpose: replay experiments for the one control fly to the other replay flies.

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq 
from mapd.sinq_builders import build_composite_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)


# Fig dir for PCA analysis
fig_dir = './Figure2'

# Load learners vs. control sinq

In [ ]:
sinq_fig1 = Sinq(sinqname='Figure1')
sinq_ctrl = Sinq(sinqname='Fig1_control')
# sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])
replay_df = sinq_ctrl.display_df().loc[sinq_ctrl.has_tag('replay')]
replay_df

In [ ]:
sinq_fig1.drop_tables()

In [ ]:
# T = sinq_fig1.restore_table(dayflycell='250924_F2_C1')
# T.extract_trial_properties()
# T.fig_folder = fig_dir
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
# fig

In [ ]:
# T = sinq_1.restore_table(dayflycell='251205_F1_C1')
# T.extract_trial_properties()
# T.fig_folder = fig_dir
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')

# Figure 2A - probe distributions

In [ ]:
refresh_table_addons()

# export the entire distribution
def call_and_export_probe_distributions(row):
    T = sinq_ctrl.restore_table(row.name)
    T.fig_folder = fig_dir
    print('Table: {}'.format(T))
    T.plot_probe_distribution(df_filter=None, index=None, bin_min=-500, bin_max=10,export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'hi'}, index=None, bin_min=-500, bin_max=10,export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'lo'}, index=None, bin_min=-500, bin_max=10,export=True)

for dfc in replay_df.index:
    row = sinq_ctrl.df.loc[dfc].copy()
    call_and_export_probe_distributions(row)
    # sinq_ctrl.drop_tables()


## Import and gather histograms

In [ ]:
import glob
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import random

folder_path = "./Figure2/exports"
fig_folder = './Figure2'

    # Color scheme for targets, adjust as needed
_force_clrs = [
    (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
    (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
    ]

def ave_and_plot_histograms(dfcs, pat,targets=None,title='None'):
    fig = Figure(figsize=(6,6), dpi=300)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    if targets:
        for key in targets.keys():
            targ_dict = targets[key]
            tgt_clr = _force_clrs[0 if targ_dict['pyasState']=='lo' else 1]
            rect = patches.Rectangle(
                (0, targ_dict['pyasXPosition']),  # (x,y) lower-left corner
                .1,                # width across counts max
                targ_dict['pyasWidth'],            # height
                edgecolor=tgt_clr,
                facecolor=tgt_clr,
                alpha=1
            )
            ax.add_patch(rect)

    normalized_counts_list = []
    probe_bins = None

    for dfc in dfcs:
        pattern=f'{dfc}'+pat
        print(pattern)
        file_path = glob.glob(os.path.join(folder_path, pattern))

        with open(file_path[0], 'rb') as f:
            data = pickle.load(f)
            bins = data['probe_bins']
            
            if probe_bins is None:
                probe_bins = bins
            else:
                assert np.allclose(probe_bins, bins), "Probe bins differ between files!"

            counts = data['total_counts']
            if '210302_F1_C2' in file_path[0]:
                print('shifting the bins for this one fly')
                print(type(counts))
                print(counts.shape)
                counts = np.roll(counts,-10,axis=0)

            norm_counts = counts / counts.sum()  # Normalize to sum to 1
            normalized_counts_list.append(norm_counts)
            ax.step(norm_counts, probe_bins[:-1], where='post', label=dfc)

    avg_norm_counts = np.mean(normalized_counts_list, axis=0)
    ax.step(avg_norm_counts, probe_bins[:-1], where='post', color='black', linewidth=2, label='Avg Normalized Histogram')

    ax.legend()
    ax.set_xlabel('Probe Position')
    ax.set_ylabel('Normalized Frequency')
    ax.grid(True, alpha=0.5)

    save_path = os.path.join(fig_folder, title + '.svg')
    fig.savefig(save_path,format = 'svg')
    print(f"Saved plot to {save_path}")
    return fig


In [ ]:
targets = sinq_ctrl.restore_table('251205_F1_C1').targets
targets

In [ ]:
dfcs = sinq_fig1.df.index
dfcs

In [ ]:
dfcs

In [ ]:
# dfcs = sinq_fig1.df.loc[sinq_fig1.has_tag('learn')]
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title= fn+ '_all_learners'))

In [ ]:
replay_df.index

In [ ]:
dfcs = replay_df.index
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title= fn+ '_all_learners'))

# Functions

In [ ]:
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

import plotly.express as px
import matplotlib.colors as mcolors
import seaborn as sns
from collections import defaultdict

k_spring_constant = 0.0829 #uN/um

def probe_velocity(t,p):
        x = p.squeeze()
        t = t.squeeze()
        v = np.gradient(x, t)
        return v

def probe_rms_velocity(t,p):
    """Weights faster movements more"""
    v = probe_velocity(t,p)
    rms_vigor = np.sqrt(np.mean(v**2))
    return rms_vigor

# def probe_jerk_energy(self):
#     """A measure of effort in motor control"""
#     ds_time = self.time[self.downsample_probe]
#     jerk = np.gradient(self.probe_acceleration(), ds_time)
#     jerk_energy = np.sum(jerk**2) * np.diff(ds_time[2:3])
#     return jerk_energy

def probe_power(t,p):
    v = probe_velocity(t,p)
    power = k_spring_constant * p * v
    return power

def probe_work(t,p):
    """Work Done Against the Spring"""
    power = probe_power(t,p)
    work = np.trapezoid(power,t)
    return work

def probe_positive_effort(t,p):
    '''Assumes p is never negative'''
    vel = probe_velocity(t,p)
    pos_vel = np.clip(vel, a_min=0, a_max=None)
    power = k_spring_constant * p * pos_vel
    effort = np.trapezoid(np.clip(power, a_min=0, a_max=None), t)
    return effort

def make_successful_trial_products(T:Table = None,fig_dir = './Figure2_folded'):
    output_directory =  f'{fig_dir}/{T.dfc}/successes_{T.dfc}'
    export_path =       f'{fig_dir}/{T.dfc}/exports'

    T.find_successful_trials()
    trials = T.df.loc[:,['Trial','success','hard_success','soft_success','pyasState']].copy()
    trials['next_trial'] = trials['Trial'].shift(-1)
    trials = trials.loc[trials['success']]
    print(trials.shape[0])

    trials = trials.loc[trials['hard_success']]
    print(trials.shape[0])

    trials['fig'] = pd.NA
    trials['v_rms'] = pd.NA
    trials['effort'] = pd.NA

    for t_num in trials.index:
        fig = Figure(figsize=(6, 6), dpi=200)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111)

        # success_index = -1
        # success_index = success_index+1
        # row = trials.iloc[success_index]
        row = trials.loc[t_num]
        trial1 = row['Trial']
        trial2 = row['next_trial']

        probe = []
        time = []
        for trial in [trial1,trial2]:
            pps = (trial.probe_position - trial.probeZero).squeeze()
            probe.append(pps[trial.downsample_probe])
            time.append(trial.time[trial.downsample_probe].squeeze())

        dT = np.diff([time[0][0],time[0][-1]])
        dt = np.diff([time[0][0],time[0][1]])
        time[1] = time[1]+dT+dt
        t = np.concat(time) #.ravel()
        p = np.concat(probe) #.ravel()

        start_time = trial1.as_duration
        start_idx = np.where(t>start_time)[0][0]

        # end_pos = probe[1][(time[1]>=0) & (time[1]<1)].mean()
        end_pos = (trial2.probe_position[(trial2.time>=0) & (trial2.time<1)].mean() - trial2.probeZero)
        # print(end_pos)
        end_idx = np.where(p<end_pos)[0][-1]
        end_time = t[end_idx]
        search_time = t[start_idx:end_idx]
        search = p[start_idx:end_idx]

        # v = probe_velocity(search_time,search)
        v_rms = probe_rms_velocity(search_time,search)
        search_power = probe_power(search_time,-search)
        search_effort = probe_positive_effort(search_time,-search)

        info_str = f'v_rms = {v_rms:.1f}; effort = {search_effort:.1f}'

        T.plot_some_probe_groups(T.df.loc[row.name:row.name+2,:].index,ax=ax) #,force_pos=True)
        x_lims = ax.get_xlim()
        y_lims = ax.get_ylim()
        ax.plot(x_lims,end_pos*np.array([1,1]))
        ax.plot(search_time,search)
        scaled_power = search_power/np.max((np.abs([search_power.max(),search_power.min()])))
        # ax.plot(search_time,-scaled_power)
        
        def tpos(lims,scalar=.9):
            return np.diff(lims)*scalar+lims[0]
        ax.text(tpos(x_lims,.6),tpos(y_lims),info_str)

        trials.loc[t_num,'fig'] = fig
        trials.loc[t_num,'v_rms'] = v_rms
        trials.loc[t_num,'effort'] = search_effort

    for trn in trials.index:
        trials.loc[trn,'fig'].savefig(f'{output_directory}/trial_{trn}.svg')

    cats = trials.index
    n = len(trials)
    palette = sns.color_palette("Set2", n)
    color_map = dict(zip(cats, palette))

    # Reuse proj_df and color_map from above
    export_df = trials[['success','hard_success','soft_success','pyasState','v_rms','effort']] # trial_number	Trial	success	hard_success	soft_success	pyasState	next_trial	fig	v_rms	effort
    export_df = export_df.reset_index()
    export_df['dfc'] = T.dfc
    pklname = f'{export_path}/{T.dfc}_{T.genotype}_success_df.pkl'
    print(pklname)
    export_df.to_pickle(pklname)
    
    plot_df = trials.reset_index()

    # Convert seaborn/matplotlib colors to hex for plotly
    # color_map = dict(zip(cats, palette))
    color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

    fig = px.scatter(
        plot_df,
        x="v_rms", y="effort", 
        color="pyasState",
        color_discrete_map=color_map_hex,
        hover_data=["trial_number",],
        title="v_rms vs. effort"
    )
    fig.update_traces(marker=dict(size=5, opacity=0.9))
    fig.update_layout(
        legend_title_text="trial_number",
        scene=dict(
            xaxis_title=f"v_rms",
            yaxis_title=f"effort",
        )
    )

    fig.update_layout(
        width=1100, height=850,                    # << size here
        margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
        font=dict(size=16),
        legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
        scene=dict(
            xaxis=dict(tickfont=dict(size=12)),
            yaxis=dict(tickfont=dict(size=12)),
            zaxis=dict(tickfont=dict(size=12)),
        ),
        hoverlabel=dict(font_size=14)
    )

    output_directory = f'{fig_dir}/{T.dfc}'
    os.makedirs(output_directory, exist_ok=True)

    fig.write_image(f'{output_directory}/success_effort_v_rms_{T.dfc}.png')


def fold_blocks(T:Table = None,fig_dir:str = './Figure2_folded',norm:bool = False):
    svg_path = f'{fig_dir}/{T.dfc}'
    output_directory = f'{fig_dir}/{T.dfc}/successes_{T.dfc}'
    export_path = f'{fig_dir}/{T.dfc}/exports'

    df = T.df.loc[T.df.index>100,['pyasState','op_cnd_blocks','as_outcome','success','as_duration']]

    df = df.copy()
    df['block_pair'] = (df['op_cnd_blocks'] - 1) // 2 + 1   # pair_id 1..14

    # 2. Align trials relative to the start of each block
    df['trial_in_block'] = df.groupby('op_cnd_blocks').cumcount()
    df['probe_as'] = (df['as_duration']>0) & (df['as_outcome']=='probe')
    df = df.loc[df.trial_in_block<50]
    
    outcome_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'as_outcome'])
        .size()
        .unstack(fill_value=0)
    )
    outcome_counts = outcome_counts.loc[['hi','lo']]
    outcome_counts['max_cnts'] = outcome_counts.sum(axis=1,skipna=True)
    outcome_counts['as_off'] = outcome_counts['as_off'] + outcome_counts['as_off_late']
    outcome_counts['no_as'] = outcome_counts['no_as_no_mv'] + outcome_counts['no_as_mv']
    outcome_counts['to'] = outcome_counts['timeout_fail'] + outcome_counts['timeout']

    success_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'success'])
        .size()
        .unstack(fill_value=0)
    )
    if not True in success_counts.columns:
        success_counts[True] = 0
    success_counts = success_counts.loc[['hi','lo'],True]
    success_counts = success_counts.to_frame(name='success')
    outcome_counts['success'] = success_counts['success']

    probe_as_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'probe_as'])
        .size()
        .unstack(fill_value=0)
    )
    probe_as_counts = probe_as_counts.loc[['hi','lo']]
    if not True in probe_as_counts.columns:
        probe_as_counts[True] = 0
    probe_as_counts = probe_as_counts.loc[['hi','lo'],True]
    probe_as_counts = probe_as_counts.to_frame(name='probe_as')
    outcome_counts['probe_as'] = probe_as_counts['probe_as']
    
    outcome_counts.to_pickle(path=f'{export_path}/folded_outcome_counts_{T.dfc}.pkl')

    # del fig
    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
              'no_as': "#000cad",
              'no_as_mv': "#6068cb",
              'success': "#009480",
              'to': "#FF0000",
              'probe_as': "#FF30FC",
            }

    hi_x = outcome_counts.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': outcome_counts.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        # nrows = len(ax_ids), ncols = 1, index = i
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = outcome_counts.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / outcome_counts.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(outcome_counts.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()

    os.makedirs(svg_path, exist_ok=True)
    fig.savefig(f'{svg_path}/outcomes_over_blocks_{T.dfc}.svg')
    
    display(fig)

# Export v_rms vs effort, and AS_outcomes over folded blocks

In [ ]:
# for dfc in ['250304_F3_C1','250425_F1_C1']:
for dfc in sinq.df.index:
    dfc_path = f'./Figure2/folded_blocks/{dfc}'
    success_path = f'{dfc_path}/successes_{dfc}'
    export_path =  f'{dfc_path}/exports'
    
    os.makedirs(dfc_path, exist_ok=True)
    os.makedirs(success_path, exist_ok=True)
    os.makedirs(export_path, exist_ok=True)
    
    table_chk_pnt = (not os.path.isfile(f'{dfc_path}/outcomes_over_blocks_{dfc}.svg') 
                 or not os.path.isfile(f'{export_path}/folded_outcome_counts_{dfc}.pkl') 
                 or not os.path.isfile(f'{export_path}/{dfc}_{sinq.df.loc[dfc,'genotype']}_success_df.pkl'))

    if table_chk_pnt:    
        T = sinq.restore_table(dfc)
        T.extract_trial_properties()
        T.find_successful_trials()
        make_successful_trial_products(T,fig_dir='./Figure2/folded_blocks/')
        fold_blocks(T,fig_dir='./Figure2/folded_blocks/')
    else:
        print(f'Files for {dfc} already exist')


    sinq.drop_tables()

# Plot v_rms vs. effort

In [ ]:
import glob

df_list = []

for dfc in reversed(sinq.df.index):
    export_path = os.path.join(f'./Figure2/{dfc}','exports')
    pklpat = f'{export_path}/{dfc}_*_success_df.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    df_list.append(pd.read_pickle(file_path))

success_df = pd.concat(df_list, axis=0)
cats = success_df.dfc.unique()
palette = sns.color_palette("Set2", len(cats))
color_map = dict(zip(cats, palette))
color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

fig = px.scatter(
    success_df,
    x="v_rms", y="effort", 
    color="dfc",
    color_discrete_map=color_map_hex,
    hover_data=['dfc',"trial_number",],
    title="v_rms vs. effort"
)
fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(
    legend_title_text="trial_number",
    scene=dict(
        xaxis_title=f"v_rms",
        yaxis_title=f"effort",
    )
)

fig.update_layout(
    width=1100, height=850,                    # << size here
    margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
    font=dict(size=16),
    legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
    scene=dict(
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        zaxis=dict(tickfont=dict(size=12)),
    ),
    hoverlabel=dict(font_size=14)
)

# output_directory = f'./Figure2/{T.dfc}'
# os.makedirs(output_directory, exist_ok=True)

fig.write_image(f'./Figure3/success_effort_v_rms_all.png')


# Fold blocks and average outcomes across blocks

In [ ]:
op_df.loc[op_df['geno_smpl']=='w']

In [ ]:
from collections import defaultdict
def _get_gstr(g):
    g = g.replace('.', '_')
    g = g.replace('>', '_')
    g = g.replace('/', '_')
    g = g.replace('[', '')
    g = g.replace(']', '')
    g = g.replace('*', '')
    return g

for g in ['w','+;iav>kir2.1','w;iav>kir2.1','+;TH>kir2.1']:

    norm = True
    total_outcomes = None
    total_success = None
    total_probe = None
    outcome_dict = defaultdict(dict)

    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        export_path = f'./Figure3/folded_blocks/{dfc}/exports'
        pklpat = f'{export_path}/folded_outcome_counts_{dfc}.pkl'
        file_path = glob.glob(pklpat)
        if file_path:
            file_path = file_path[0]
            print(file_path)

        outcome_dict[dfc] = pd.read_pickle(file_path)

        if total_outcomes is None:
            total_outcomes = outcome_dict[dfc].copy()
        else:
            total_outcomes = total_outcomes.add(outcome_dict[dfc], fill_value=0)

    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
                'no_as': "#000cad",
                'no_as_mv': "#6068cb",
                'success': "#009480",
                'to': "#FF0000",
                'probe_as': "#FF30FC",
            }

    hi_x = total_outcomes.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': total_outcomes.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    # for dfc in reversed(sinq2.df.index):
    for dfc in outcome_dict.keys():
        for outcome in outcome_to_ax.keys():
            for st in ['hi','lo']:
                outcomes = outcome_dict[dfc]
                ax = ax_id_to_ax[outcome_to_ax[outcome]]
                hi_y = outcomes.xs(st, level='pyasState').get(outcome)
                if norm:
                    hi_y = hi_y / outcomes.xs(st, level='pyasState').get('max_cnts')

                # ax.scatter(
                #             x[st], hi_y, marker='.',s=5, alpha=0.9, c=colors[outcome]
                #         )
                if norm:
                    ax.set_ylim([0, 1])
                else:
                    ax.set_ylim([0, np.max(outcomes.xs(st, level='pyasState').get('max_cnts'))])


    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = total_outcomes.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / total_outcomes.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(total_outcomes.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()
    fig.savefig(f'./Figure2/outcomes_over_blocks_{_get_gstr(g)}.svg',format='svg')
    display(fig)

In [ ]:
pd.Index([i for i in range(500)])

In [ ]:
# index=
T.df.loc[T.df.index[50:450]]

In [ ]:
sinq.drop_tables()

In [ ]:
for g in ['w','+;iav>kir2.1','+;TH>kir2.1']:
    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        T = sinq.restore_table(dfc)
        T.extract_trial_properties()
        T.fig_folder = f'./Figure3/heatmaps/{_get_gstr(g)}'
        idx=T.df.loc[T.df.index[50:450]].index
        print(f'{dfc}: making heat map for {len(idx)} trials')
        try:
            fig,ax,df = T.plot_probe_position_heatmap(index=idx,cmin=-500,cmax=10,format='svg')
        except Exception as e:
            print(f'{e}')